# 최종 집계 — 학교별 댓글 수

`gt_match.ipynb`가 저장한 `data/processed/gt_match_results.csv`의 `ans` 컬럼(댓글당 매칭된
정식 학교명, 여러 개면 공백 구분)을 펼쳐서 학교별로 카운트.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src"))

REPO_ROOT

WindowsPath('d:/Study/dongguk_university/dreampath')

In [ ]:
import pandas as pd

df = pd.read_csv(
    REPO_ROOT / "data" / "processed" / "gt_match_results.csv", encoding="utf-8-sig"
)
df["gt_match"] = df["gt_match"].fillna("")
df.shape

## ans를 펼쳐서 학교별 댓글 수 집계

한 댓글에 학교가 2개 이상 언급된 경우(`ans_count >= 2`), 시간 관계상 어느 쪽이 진짜 정답인지
판단하는 로직은 만들지 못해서 **둘 다 그대로 카운트**함. 그래서 아래 집계 총합은 1,000건보다
클 수 있음 — 이건 알려진 한계점.

In [ ]:
# gt_match에 학교가 공백으로 여러 개 있으면(예: "잠실초등학교 대치초등학교") 각각 +1씩 카운트
all_schools = [school for match in df["gt_match"] for school in match.split()]

school_counts = pd.Series(all_schools).value_counts().sort_values(ascending=False)
school_counts.index.name = "학교명"
school_counts.name = "count"
school_counts

In [ ]:
out_path = REPO_ROOT / "outputs" / "ans" / "result.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
school_counts.to_csv(out_path, encoding="utf-8-sig")
out_path